In [1]:
pip install opencv-python numpy scikit-image scikit-learn joblib matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import cv2
import numpy as np
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib
import matplotlib.pyplot as plt

# Paths
jaundice_path = 'dataset/jaundiced'
healthy_path = 'dataset/healthy'

data = []
labels = []

# Feature extraction
def extract_features(img):
    img = cv2.resize(img, (128, 128))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    features, _ = hog(gray, orientations=9, pixels_per_cell=(8, 8),
                      cells_per_block=(2, 2), block_norm='L2-Hys', visualize=True)
    return features

# Load jaundiced images
jaundiced_count = 0
for img_name in os.listdir(jaundice_path):
    path = os.path.join(jaundice_path, img_name)
    img = cv2.imread(path)
    if img is not None:
        data.append(extract_features(img))
        labels.append(1)
        jaundiced_count += 1

# Load healthy images
healthy_count = 0
for img_name in os.listdir(healthy_path):
    path = os.path.join(healthy_path, img_name)
    img = cv2.imread(path)
    if img is not None:
        data.append(extract_features(img))
        labels.append(0)
        healthy_count += 1

# Convert to arrays
X = np.array(data)
y = np.array(labels)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# SVM training
model = SVC(kernel='linear', probability=True)
model.fit(X_train, y_train)

# Save model
os.makedirs("model", exist_ok=True)
joblib.dump(model, 'model/jaundice_model.pkl')

# Accuracy
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

# Save stats
os.makedirs("static", exist_ok=True)
with open("static/counts.txt", "w") as f:
    f.write(f"{healthy_count},{jaundiced_count},{acc:.2f}")

# Accuracy bar plot
plt.figure(figsize=(4, 3))
plt.bar(['Accuracy'], [acc], color='lightgreen')
plt.title("Model Accuracy")
plt.ylim([0, 1])
plt.tight_layout()
plt.savefig("static/accuracy.png")
plt.close()

print(f"Training complete. Accuracy: {acc:.2f}")

# Bar chart for image count
plt.figure(figsize=(4, 3))
plt.bar(['Healthy', 'Jaundiced'], [healthy_count, jaundiced_count], color=['lightblue', 'salmon'])
plt.title("Image Count Distribution")
plt.tight_layout()
plt.savefig("static/image_counts.png")
plt.close()

# Pie chart
plt.figure(figsize=(4, 4))
plt.pie([healthy_count, jaundiced_count], labels=['Healthy', 'Jaundiced'], autopct='%1.1f%%', colors=['lightblue', 'salmon'])
plt.title("Dataset Split")
plt.savefig("static/image_pie.png")
plt.close()
